# Mini-GPT — Entraînement sur Google Colab

Ce notebook clone le repo et lance l'entraînement directement.

**Pré-requis :** Runtime > Change runtime type > **GPU** (T4 ou mieux)

## 1. Clone du repo et installation

In [ ]:
!git clone -b transformers https://github.com/eric-houzelle/mini-gpt.git
%cd mini-gpt
!pip install -q -r requirements.txt

## 2. Vérification GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Pas de GPU. Allez dans Runtime > Change runtime type > GPU.")

## 3. Configuration

On écrit le `.env` et le `config.json` directement. Modifiez les valeurs ci-dessous selon vos besoins.

In [ ]:
%%writefile .env
DATASET_NAME=Houzeric/wiki-FR-dataset
DATASET_KEY=text
EVAL_PROMPT="Un atome est"
TOKENIZER_NAME=camembert-base
MODEL_SAVE_PATH=checkpoints/best_miniGPT.pt
EVAL_EVERY_STEPS=500

In [ ]:
%%writefile config.json
{
  "training": {
    "num_epochs": 300,
    "batch_size": 32,
    "learning_rate": 8e-4,
    "warmup": 20000,
    "override_lr": null,
    "override_best_loss": null
  },
  "model": {
    "embed_dim": 640,
    "depth": 14,
    "heads": 10,
    "block_size": 256,
    "dropout": 0.05,
    "hidden_dim": 3840,
    "weight_sharing": "none",
    "use_rope": true,
    "use_gradient_checkpointing": false
  },
  "data": {
    "max_texts": 500000,
    "train_split_ratio": 0.90
  }
}

## 4. Lancer l'entraînement

In [ ]:
!mkdir -p checkpoints
!python train.py

## 5. Tester le modèle

In [ ]:
!python generatev2.py --prompt "La France est un pays" --tokens 200 --temperature 0.7

## 6. Sauvegarder le checkpoint sur Google Drive (optionnel)

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

# import shutil, os
# drive_path = "/content/drive/MyDrive/mini-gpt-checkpoints/"
# os.makedirs(drive_path, exist_ok=True)
# shutil.copy("checkpoints/best_miniGPT.pt", drive_path)
# print(f"Checkpoint copié vers {drive_path}")

## 7. Reprendre l'entraînement depuis Google Drive (optionnel)

Si vous avez sauvegardé un checkpoint sur Drive, vous pouvez le restaurer pour reprendre l'entraînement.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

# import shutil, os
# drive_ckpt = "/content/drive/MyDrive/mini-gpt-checkpoints/best_miniGPT.pt"
# if os.path.exists(drive_ckpt):
#     os.makedirs("checkpoints", exist_ok=True)
#     shutil.copy(drive_ckpt, "checkpoints/best_miniGPT.pt")
#     print("Checkpoint restauré depuis Drive.")
# else:
#     print("Pas de checkpoint trouvé sur Drive.")

# # Puis relancez l'entraînement :
# # !python train.py